**Comentario de entrada:**
> Este bloque prepara el entorno para poder importar el paquete del proyecto (`src`) desde el notebook, ya que este vive en `notebooks/` y `src` está un nivel arriba. Primero agrega la raíz del proyecto al `sys.path` si aún no está incluida, y luego importa `get_adult_census_data`, la función que centraliza en `data.py` la descarga del dataset Adult Census Income desde Kaggle (vía `kagglehub`) y su copia a `data/raw`. Finalmente llama a la función y guarda en `csv_path` la ruta al archivo `.csv` resultante.

In [8]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.inf8239_u01.data import get_adult_census_data

csv_path = get_adult_census_data()
print(csv_path)

C:\Users\Administrator\Desktop\Maestria Big Data\11_INF-8239_Ciencias_de_Datos_II\01_Unidad\INF8239_U01\data\raw\adult.csv


**Comentario de salida:**
> Al ejecutarse, `csv_path` queda con la ruta absoluta al archivo `.csv` del dataset Adult Census Income dentro de `data/raw`. Si es la primera ejecución, el archivo se descarga y se copia a esa carpeta; si ya existe (ejecuciones posteriores), `kagglehub` reutiliza su caché y la copia se repite sin error. El `print(csv_path)` confirma la ubicación exacta del archivo para usarlo en la siguiente celda con `pd.read_csv(csv_path)`.

**Comentario de entrada:**
> Este bloque carga en memoria el dataset descargado en el paso anterior (`csv_path`, la ruta al CSV de Adult Census Income en `data/raw`) y hace un primer reconocimiento de su estructura: cuántas filas y columnas tiene (`shape`), qué tipo de dato almacena cada columna (`dtypes`), y cómo lucen sus primeras y últimas filas (`head`, `tail`), para detectar a simple vista formatos inesperados, encabezados corridos o filas vacías al final del archivo.

In [9]:
import pandas as pd

df = pd.read_csv(csv_path)
print(df.shape)
print(df.dtypes)
print(df.head())
print(df.tail())
assert not df.empty

(32561, 15)
age               int64
workclass           str
fnlwgt            int64
education           str
education.num     int64
marital.status      str
occupation          str
relationship        str
race                str
sex                 str
capital.gain      int64
capital.loss      int64
hours.per.week    int64
native.country      str
income              str
dtype: object
   age workclass  fnlwgt     education  education.num marital.status  \
0   90         ?   77053       HS-grad              9        Widowed   
1   82   Private  132870       HS-grad              9        Widowed   
2   66         ?  186061  Some-college             10        Widowed   
3   54   Private  140359       7th-8th              4       Divorced   
4   41   Private  264663  Some-college             10      Separated   

          occupation   relationship   race     sex  capital.gain  \
0                  ?  Not-in-family  White  Female             0   
1    Exec-managerial  Not-in-family  White  F

**Comentario de salida:**
> Al ejecutarse, `df` queda como un DataFrame de pandas con todas las filas y columnas del dataset cargadas en memoria. Los `print` muestran en pantalla las dimensiones del dataset, el tipo de cada columna (`int64`, `object`, etc.) y una vista previa del inicio y el final de los datos. El `assert not df.empty` no imprime nada si todo está bien; si el archivo se descargó vacío o corrupto, detiene la ejecución con un `AssertionError`, evitando seguir trabajando sobre datos inválidos.

**Comentario de entrada:**
> Este bloque construye una tabla de auditoría del dataset (`audit`) que resume, columna por columna, el tipo de dato, la cantidad y el porcentaje de valores ausentes, y el número de valores únicos (incluyendo nulos, gracias a `dropna=False`). Ordena la tabla de mayor a menor porcentaje de ausentes para identificar rápido qué columnas necesitan más atención en el preprocesamiento. Además cuenta cuántas filas están completamente duplicadas en `df`, un chequeo básico de calidad antes de definir el target.

In [10]:
audit = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "ausentes": df.isna().sum(),
    "porcentaje_ausente": (df.isna().mean()*100).round(2),
    "unicos": df.nunique(dropna=False)
}).sort_values("porcentaje_ausente", ascending=False)
print("Duplicados:", df.duplicated().sum())
display(audit)

Duplicados: 24


,tipo,ausentes,porcentaje_ausente,unicos
age,int64,0,0.0,73
workclass,str,0,0.0,9
fnlwgt,int64,0,0.0,21648
education,str,0,0.0,16
education.num,int64,0,0.0,16
marital.status,str,0,0.0,7
occupation,str,0,0.0,15
relationship,str,0,0.0,6
race,str,0,0.0,5
sex,str,0,0.0,2


**Comentario de salida:**
> Al ejecutarse, `audit` queda como un DataFrame con una fila por cada columna del dataset original y cuatro columnas descriptivas: `tipo`, `ausentes`, `porcentaje_ausente` y `unicos`, ordenado de la columna con más valores faltantes a la que tiene menos. El `print("Duplicados:", ...)` muestra en pantalla cuántas filas completas están repetidas en `df`. El `display(audit)` renderiza la tabla completa para inspección visual, dejando en evidencia qué columnas requieren imputación, si hay filas duplicadas que valga la pena eliminar, y qué tan diversa es cada variable (columnas con `unicos` muy bajo podrían ser constantes; con `unicos` muy alto, posibles identificadores).

**Comentario de entrada:**
> Este bloque define cuál es la variable objetivo (`TARGET = "income"`) y separa el dataset en las variables predictoras (`X`) y el target (`y`), retirando además cualquier columna en `DROP_COLUMNS` que se considere una fuga de información o un identificador sin valor predictivo. También verifica que la columna elegida como target exista en el DataFrame antes de continuar.

In [11]:
TARGET = "income"
DROP_COLUMNS = []  # deja [] si no aplica

assert TARGET in df.columns
X = df.drop(columns=[TARGET] + DROP_COLUMNS)
y = df[TARGET]
print(y.value_counts(dropna=False))
assert y.notna().all()
assert y.nunique() >= 2

income
<=50K    24720
>50K      7841
Name: count, dtype: int64


**Comentario de salida:**
> Al ejecutarse, `X` queda como un DataFrame con todas las columnas predictoras (sin `income` ni las columnas de `DROP_COLUMNS`), y `y` como una Serie con los valores de `income` para cada fila. El `print(y.value_counts(dropna=False))` muestra en pantalla cuántas filas hay de cada clase (`<=50K` y `>50K`), permitiendo ver si el target está desbalanceado. El `assert y.notna().all()` confirma que no hay valores nulos en el target, y `assert y.nunique() >= 2` confirma que hay al menos dos clases distintas; si cualquiera de las dos condiciones falla, la ejecución se detiene con un `AssertionError` antes de intentar entrenar un modelo sobre un target inválido.

**Comentario de entrada:**
> Este bloque identifica automáticamente qué columnas de `X` son numéricas (`num_cols`) y cuáles son categóricas (`cat_cols`), y arma un `ColumnTransformer` que aplicará un tratamiento distinto a cada grupo: a las numéricas les imputa la mediana en los valores faltantes y las estandariza (media 0, desviación 1); a las categóricas les imputa la categoría más frecuente y las convierte a variables dummy con `OneHotEncoder`, ignorando categorías nuevas que puedan aparecer en datos futuros. Empaquetar esto en un `Pipeline`/`ColumnTransformer` (en vez de transformar `X` directamente) evita fugas de información, porque el ajuste de imputación y escalado se hará solo con los datos de entrenamiento en el paso siguiente.

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler())])
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                     ("onehot", OneHotEncoder(handle_unknown="ignore"))])

preprocess = ColumnTransformer([("num", num_pipe, num_cols),
                                ("cat", cat_pipe, cat_cols)])
print(len(num_cols), len(cat_cols))

6 8


**Comentario de salida:**
> Al ejecutarse, `num_cols` y `cat_cols` quedan como listas con los nombres de las columnas numéricas y categóricas de `X`, respectivamente. `preprocess` queda como un objeto `ColumnTransformer` ya configurado (pero todavía sin ajustar a datos), listo para integrarse dentro de un `Pipeline` de modelado. El `print(len(num_cols), len(cat_cols))` muestra en pantalla cuántas columnas cayeron en cada grupo, útil para confirmar que la separación automática por tipo de dato coincide con lo que esperas del dataset (por ejemplo, que variables como `age` o `hours-per-week` queden como numéricas y `workclass` o `occupation` como categóricas).

**Comentario de entrada:**
> Este bloque separa los datos en conjunto de entrenamiento y prueba (80/20), manteniendo la misma proporción de clases del target en ambos subconjuntos gracias a `stratify=y`, y usando `random_state=42` para que la división sea reproducible. Luego arma dos pipelines completos: un modelo `dummy` que siempre predice la clase más frecuente (línea base ingenua) y una `svm` con kernel por defecto, ambos incluyendo el `preprocess` del paso anterior para que el ajuste de imputación, escalado y codificación se haga únicamente con los datos de entrenamiento, sin tocar el conjunto de prueba durante el ajuste.

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, f1_score

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.20, random_state=42, stratify=y)

dummy = Pipeline([("prep", preprocess), ("model", DummyClassifier(strategy="most_frequent"))])
svm = Pipeline([("prep", preprocess), ("model", SVC(C=1, gamma="scale", probability=True, random_state=42))])

for name, model in {"dummy": dummy, "svm": svm}.items():
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    print(name, f1_score(yte, pred, average="macro"))

print(classification_report(yte, svm.predict(Xte)))

dummy 0.4315761913073835


c:\Users\Administrator\Desktop\Maestria Big Data\11_INF-8239_Ciencias_de_Datos_II\01_Unidad\INF8239_U01\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


svm 0.7834571457689492
              precision    recall  f1-score   support

       <=50K       0.88      0.94      0.91      4945
        >50K       0.75      0.59      0.66      1568

    accuracy                           0.85      6513
   macro avg       0.82      0.76      0.78      6513
weighted avg       0.85      0.85      0.85      6513



**Comentario de salida:**
> Al ejecutarse, `Xtr`, `Xte`, `ytr`, `yte` quedan como los conjuntos de entrenamiento y prueba (features y target por separado). `dummy` y `svm` quedan ambos ajustados (`fit`) sobre `Xtr`/`ytr`, con sus preprocesadores internos ya aprendidos solo de esos datos. El bucle imprime en pantalla el F1 macro de cada modelo sobre el conjunto de prueba, permitiendo comparar de entrada si la SVM realmente supera a la línea base ingenua. El `classification_report` final muestra precisión, recall y F1 por clase para la SVM, dando una primera vista de en qué clase falla más el modelo antes de cualquier ajuste de hiperparámetros.